# Qwen3-VL Research — Colab Runner

**One-click evaluation on a Colab GPU.** Code lives on GitHub, model weights live in Google Drive, results are pushed back to GitHub.

### Before your first run (one-time setup)
1. **Runtime → Change runtime type → GPU (T4) → Save.**
2. **Add a GitHub token so Colab can push results back:**
   - On GitHub: Settings → Developer settings → Personal access tokens → *Fine-grained tokens* → generate one with **Contents: Read and write** access to the `Qwen3-VL` repo.
   - In Colab: click the **🔑 key icon** in the left sidebar → **Add new secret** → Name it `GITHUB_TOKEN`, paste the token, and toggle **Notebook access** on.

### Every session
**Runtime → Run all.** That's it. The cells, top-to-bottom:
1. Read config + GitHub token
2. Mount Google Drive (model persists here)
3. Clone or pull the latest repo from GitHub
4. Install Python dependencies
5. Download model weights to Drive (first session only, ~5 min)
6. Verify GPU + `local_transformers` wiring
7. **Run `evaluate.py`**
8. Push results back to GitHub (so they appear in your local repo on `git pull`)

---
### How data moves
```
local repo --git push--> GitHub --git pull--> Colab   (your code edits reach the GPU)
Colab runs evaluate.py -> results/<timestamp>/
Colab --git push--> GitHub --git pull--> local repo   (results come back to you)
```
- **Code** (your edits to `local_transformers/`, experiments): travels both ways through GitHub.
- **Results**: Colab commits + pushes them; you run `git pull` locally to see them.
- **Model weights** (4.5 GB): Drive only — too big for GitHub, and they persist so you only download once.

In [ ]:
# Cell 0 — Config (the only cell you might edit)
import os

# --- Repo ---
GITHUB_USER = "adikothuri3"
REPO_NAME   = "Qwen3-VL"
GIT_BRANCH  = "main"

# --- Model ---
MODEL_REPO_ID = "Qwen/Qwen3-VL-2B-Instruct"
MODEL_PATH    = "/content/drive/MyDrive/Qwen3-VL-models/Qwen3-VL-2B-Instruct"

# --- Evaluation args (passed to src/evaluate.py) ---
EVAL_ARGS = {
    "--device":         "cuda",
    "--dtype":          "float16",
    "--max-new-tokens": "64",
    "--num-samples":    "1",
}
EVAL_FLAGS = ["--no-torch-profiler"]  # drop this flag to enable the full torch.profiler pass

# --- Push results back to GitHub when the run finishes? ---
PUSH_RESULTS = True

# --- GitHub token ---
# Resolved in this order, so the notebook works from the Colab web UI *or* a
# connected kernel (VS Code / local Jupyter) where Colab Secrets are unreachable:
#   1. Colab Secrets (userdata) — only works when run from the colab.research.google.com UI
#   2. GITHUB_TOKEN environment variable — set this when running outside the Colab UI
GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    print("GitHub token loaded from Colab Secrets.")
except ImportError:
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
    print("Not running in Colab; "
          + ("loaded GITHUB_TOKEN from environment." if GITHUB_TOKEN
             else "no GITHUB_TOKEN env var set — pushing results will be skipped."))
except Exception as e:
    # e.g. NotebookAccessError (secret exists but access toggle off),
    # SecretNotFoundError, or TimeoutException (not run from the Colab UI).
    GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
    if GITHUB_TOKEN:
        print(f"Colab Secrets unavailable ({type(e).__name__}); "
              "loaded GITHUB_TOKEN from environment instead.")
    else:
        print(f"Could not read GITHUB_TOKEN from Colab Secrets ({type(e).__name__}: {e}). "
              "Set a GITHUB_TOKEN env var, or run this notebook from the Colab UI with "
              "the secret's 'Notebook access' toggled on. Pushing results will be skipped.")

REPO_DIR = f"/content/{REPO_NAME}"
# Authenticated URL is used only in subprocess calls, never printed.
_auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
REPO_URL_AUTH = f"https://{_auth}github.com/{GITHUB_USER}/{REPO_NAME}.git"
REPO_URL_SAFE = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
print(f"Repo: {REPO_URL_SAFE}  (branch: {GIT_BRANCH})")

In [5]:
# Cell 1 — Mount Google Drive (model weights persist here across sessions)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# Cell 2 — Clone or pull the latest repo from GitHub
import os, subprocess

def _git(args, **kw):
    """Run a git command; never echoes the token-bearing URL."""
    r = subprocess.run(["git", *args], capture_output=True, text=True, **kw)
    out = (r.stdout + r.stderr)
    if GITHUB_TOKEN:
        out = out.replace(GITHUB_TOKEN, "***")
    return r.returncode, out.strip()

if os.path.exists(f"{REPO_DIR}/.git"):
    _git(["-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL_AUTH])
    code, out = _git(["-C", REPO_DIR, "pull", "origin", GIT_BRANCH])
else:
    code, out = _git(["clone", "--branch", GIT_BRANCH, REPO_URL_AUTH, REPO_DIR])
print(out or "(no output)")

os.chdir(REPO_DIR)
print(f"\nWorking dir: {os.getcwd()}")
assert code == 0, "git clone/pull failed — check the output above."

KeyboardInterrupt: 

In [ ]:
# Cell 3 — Install dependencies
import subprocess, sys

r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    capture_output=True, text=True,
)
out = (r.stdout + r.stderr).strip()
print(out[-2000:] if len(out) > 2000 else out or "All packages installed.")
if r.returncode != 0:
    print("\n[warning] pip returned a non-zero exit code — review the output above.")

All packages installed.


In [ ]:
# Cell 4 — Download model weights to Drive (first session only, ~4.5 GB / ~5 min)
import os

if os.path.isdir(MODEL_PATH) and os.listdir(MODEL_PATH):
    print(f"Model already in Drive — skipping download.\n{MODEL_PATH}")
else:
    print(f"Downloading {MODEL_REPO_ID} to Drive (~4.5 GB) ...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=MODEL_REPO_ID,
        local_dir=MODEL_PATH,
        ignore_patterns=["*.pt", "*.bin"],  # safetensors only
    )
    print("\nDownload complete.")

Model already in Drive — skipping download.
/content/drive/MyDrive/Qwen3-VL-models/Qwen3-VL-2B-Instruct


In [ ]:
# Cell 5 — Verify GPU and local_transformers wiring
import torch, sys, inspect

print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU            : {props.name}")
    print(f"VRAM           : {props.total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU. Runtime -> Change runtime type -> GPU (T4) -> Save, then Run all.")

# evaluate.py wires this itself; we verify here so a misconfig fails loudly and early.
sys.path.insert(0, REPO_DIR)
import local_transformers
sys.modules["transformers"] = local_transformers
from local_transformers.models.qwen3_vl.modeling_qwen3_vl import Qwen3VLForConditionalGeneration

src = inspect.getfile(Qwen3VLForConditionalGeneration)
print(f"\nModel source   : {src}")
assert f"{REPO_DIR}/local_transformers" in src, "ERROR: not using local source!"

CUDA available : True
GPU            : Tesla T4
VRAM           : 15.6 GB

Model source   : /content/Qwen3-VL/local_transformers/models/qwen3_vl/modeling_qwen3_vl.py


In [ ]:
# Cell 6 — Run evaluate.py  (the main event)
import subprocess, sys

cmd = [sys.executable, "src/evaluate.py", "--model-id", MODEL_PATH]
for k, v in EVAL_ARGS.items():
    cmd += [k, v]
cmd += EVAL_FLAGS

print("Running:", " ".join(cmd), "\n")
# capture_output unset -> output streams live into the cell.
proc = subprocess.run(cmd)
print(f"\nProcess exited with code {proc.returncode}")
assert proc.returncode == 0, "evaluate.py failed — review the log above."

Running: /usr/bin/python3 src/evaluate.py --model-id /content/drive/MyDrive/Qwen3-VL-models/Qwen3-VL-2B-Instruct --device cuda --dtype float16 --max-new-tokens 64 --num-samples 1 --no-torch-profiler 


Process exited with code 0


In [ ]:
# Cell 7 — Push results back to GitHub
# Commits the newest results/<timestamp>/ dir and pushes it so it lands in your
# local repo on the next `git pull`. Large Chrome traces (trace_*.json) are
# gitignored, so only the JSON metrics + chart are pushed.
import os, glob

if not PUSH_RESULTS:
    print("PUSH_RESULTS is False — skipping.")
elif not GITHUB_TOKEN:
    print("No GITHUB_TOKEN — skipping push. Results remain in Colab at results/ only.")
else:
    runs = sorted(glob.glob(f"{REPO_DIR}/results/*/"))
    if not runs:
        print("No results/ runs found — nothing to push.")
    else:
        latest = runs[-1]
        _git(["-C", REPO_DIR, "config", "user.name",  "colab-runner"])
        _git(["-C", REPO_DIR, "config", "user.email", "colab@users.noreply.github.com"])
        _git(["-C", REPO_DIR, "add", "results/"])
        run_name = os.path.basename(latest.rstrip("/"))
        code, out = _git(["-C", REPO_DIR, "commit", "-m", f"Colab run: results/{run_name}"])
        print(out or "(nothing to commit)")
        if code == 0:
            code, out = _git(["-C", REPO_DIR, "push", "origin", GIT_BRANCH])
            print(out or "(push complete)")
            print("\nDone. Run `git pull` in your local repo to get these results."
                  if code == 0 else "\nPush failed — check token permissions.")
        else:
            print("Nothing new to commit (results may already be pushed).")

No GITHUB_TOKEN — skipping push. Results remain in Colab at results/ only.
